# 050 — Set up MSA runs for the 3s & 5s mdof case-study structures

Wires the Multiple-Stripe-Analysis (MSA) run files into the **existing** per-site
`mdof/` analysis folders built by notebook 011, for **both** the 3-storey and
5-storey structures, at `DEST_ROOT/site_{ii}/{n}s/mdof/`
(`DEST_ROOT = wp1_casestudy_sites`).

Each structure gets only the stripe selection pickles for **its own middle 4
IMLs**: the per-structure IML list comes from
`data_processed/05_gcim_distributions/imls_for_selection_AvgSA_03.json`, and the
lowest and highest levels (the highest is sometimes `null`) are skipped for now —
so `imls[1:-1]`.

This notebook does **not** design structures or run modal analyses. It only adds
the MSA files (`run_msa_site.py`, the max-parallel coordinator + worker,
injection/recorder helpers, and one `config_msa_{IM}.py` + `{IM}/` stripe-pickle
folder), then writes **per-storey batch launchers** (one for 3s, one for 5s).

Maximum parallelisation: `run_msa_site.py` -> `run_batch_msa_per_stripe_record.py`
(dispatches every `(stripe, record)` pair at once) -> `run_msa_per_record.py` worker.

## Dependencies

Run notebook 011 first: it builds the `mdof/` folders and their `structural_model.py`.
Any (site, storey) whose `structural_model.py` is missing is **warned and skipped**
(and excluded from the batch files).

## Filtering

Set `FILTER_TO_COMPLETE_SITES` at the top: when `True` the batch launchers list only
the sites flagged complete by the record-availability audit
(`data_processed/06_gm_selection/complete_site_im_sets.csv`). The MSA files are
copied into every existing folder regardless — the filter only affects the batch
listings.


In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import json
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_file,
    copy_analysis_config,
    configure_batch_run_file,
)
from phd_project.scripts.WP1_ground_motion_set.gm_selection import stripe_pickle_path

cfg = config.load_config()

# --- parameters -----------------------------------------------------------
DEST_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]   # site_{ii}/{n}s/mdof/
GM_SET = "AvgSA_03"
STOREYS = [3, 5]                     # build MSAs for both the 3s and 5s structures
STRIPE_ORDER_ASCENDING = True
MAX_N_RECORDS = None                 # cap records per stripe (None = all)

# When True the per-storey batch launchers list only the completed sites (from
# complete_site_im_sets.csv). MSA files are copied into every existing folder
# regardless; this only affects the batch listings.
FILTER_TO_COMPLETE_SITES = True

# mdof EDP recorder template (roof drift)
RECORDER_KEY = "ida_process_recorder_roof_drift"

RECORD_FOLDER = Path(cfg["results"][f"{GM_SET}_record_selection"])
IMLS_JSON = Path(cfg["proc_data"][f"{GM_SET}_imls_for_selection"])
# Batch launchers land under BATCH_BASE/{n}s/mdof/ (matches nb 011); the folder
# path encodes the storeys + system, so the filename is just msa_{IM}.py.
BATCH_BASE = Path(cfg["scripts"]["batch_run_analyses"]) / "casestudy_sites"
print(f"DEST_ROOT = {DEST_ROOT}")

## 1. Read the completed-site list

`ready_sites` — the site indices flagged complete for `GM_SET` by the
record-availability audit. Only used to filter the batch launchers when
`FILTER_TO_COMPLETE_SITES` is `True`.

In [3]:
ready_csv = Path(cfg["proc_data"]["gm_selection"]) / "complete_site_im_sets.csv"
ready_df = pd.read_csv(ready_csv)

ready_sites: set[int] = {
    int(row["site"]) for _, row in ready_df.iterrows() if str(row["im"]) == GM_SET
}

print(f"{len(ready_sites)} completed sites for {GM_SET}  (from {ready_csv.name})")

37 completed sites for AvgSA_03  (from complete_site_im_sets.csv)


## 2. Per-structure IMLs (middle 4)

The per-structure IML list comes from `imls_for_selection_{GM_SET}.json`, keyed by
site index then structure tag `"{n}s_cbf_dc2_site{ii}"` (6 levels, lowest -> highest,
the highest sometimes `null`). `middle_imls` drops the lowest and highest and returns
the middle 4. The sites to set up are the JSON's site keys.

In [4]:
with open(IMLS_JSON) as f:
    imls_data = json.load(f)

# sites to set up = the site keys in the IML json
all_sites = sorted(int(k) for k in imls_data)


def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"


def middle_imls(site_idx: int, n: int) -> list[float]:
    """Middle 4 IMLs for a (site, storey) structure: drop the lowest and the
    highest (the highest is sometimes null)."""
    lst = imls_data[str(site_idx)][structure_tag(site_idx, n)]
    return [x for x in lst[1:-1] if x is not None]


print(f"{len(all_sites)} sites in {IMLS_JSON.name}")
for n in STOREYS:
    example = middle_imls(all_sites[0], n)
    print(f"  {n}s example (site {all_sites[0]}): {example}")

60 sites in imls_for_selection_AvgSA_03.json
  3s example (site 0): [0.31, 0.36, 0.4, 0.5]
  5s example (site 0): [0.285, 0.33, 0.36, 0.4]


## 3. `add_msa_files` helper

Copies the MSA run files into a `mdof/` folder and creates one `config_msa_{IM}.py`
+ `{IM}/` stripe-pickle subfolder holding only the structure's **middle 4** stripe
pickles (resolved via `stripe_pickle_path`). Copies the `run_msa_per_record.py`
worker required by the max-parallel coordinator. Returns any expected-but-missing
stripe pickles so they can be reported.

In [5]:
def add_msa_files(folder: Path, site_idx: int, n: int) -> list[Path]:
    """Add MSA run/coordinator/worker/injection/recorder files + one config and a
    stripe-pickle subfolder holding this structure's middle-4 stripes. Returns the
    list of expected stripe pickles that were missing at source."""
    copy_file(cfg["templates"]["run_msa_site"], folder / "run_msa_site.py")
    copy_file(cfg["templates"]["run_batch_msa_per_stripe_record"],
              folder / "run_batch_msa_per_stripe_record.py")
    copy_file(cfg["templates"]["run_msa_per_record"],
              folder / "run_msa_per_record.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"][RECORDER_KEY],
              folder / "msa_process_recorders.py")

    record_src = Path(cfg["proc_data"]["gm_records"]).as_posix()

    gm_dir = folder / GM_SET
    gm_dir.mkdir(parents=True, exist_ok=True)
    # clear any previously-copied stripe pickles first so stale / old-scheme names
    # don't linger alongside the current middle-4 set (find_stripe_pickles globs
    # all *stripe*gm_selection* in the folder).
    for old in gm_dir.glob("*__stripe_*__gm_selection.pickle"):
        old.unlink()

    missing: list[Path] = []
    for iml in middle_imls(site_idx, n):
        src = stripe_pickle_path(RECORD_FOLDER, site_idx, iml)
        if src.exists():
            copy_file(src, gm_dir / src.name)
        else:
            missing.append(src)

    copy_analysis_config(
        cfg["templates"]["config_msa"],
        folder / f"config_msa_{GM_SET}.py",
        results_folder_name=f"msa_{GM_SET}",
        gm_selection_src_str=gm_dir.as_posix(),
        record_src_str=record_src,
        stripe_order_ascending=STRIPE_ORDER_ASCENDING,
        max_n_records=MAX_N_RECORDS,
    )
    return missing

## 4. Add MSA files to the 3s & 5s mdof folders

For every site and each storey, add the MSA files to the `mdof/` folder built by
notebook 011. Missing folders / structural models are warned and skipped (and are
excluded from the batch files). Files are added for every existing folder,
regardless of the completed-site filter.

In [9]:
# storey -> {site_idx: folder}, only (site, storey) successfully set up (model present)
built: dict[int, dict[int, Path]] = {n: {} for n in STOREYS}
skipped: list[tuple[int, int]] = []
missing_stripes: list[Path] = []

for site_idx in all_sites:
    for n in STOREYS:
        folder = DEST_ROOT / f"site_{site_idx}" / f"{n}s" / "mdof"
        if not (folder / "config_structural_model.py").exists():
            print(f"WARNING: skipping site {site_idx} [{n}s]: "
                  f"{folder / 'config_structural_model.py'} not found (run nb 011 first)")
            skipped.append((site_idx, n))
            continue
        missing_stripes += add_msa_files(folder, site_idx, n)
        built[n][site_idx] = folder

for n in STOREYS:
    print(f"{n}s: MSA files added for {len(built[n])} sites")
if skipped:
    print(f"skipped {len(skipped)} (site, storey) pair(s): {skipped}")
if missing_stripes:
    print(f"WARNING: {len(missing_stripes)} expected stripe pickle(s) missing at source, "
          f"e.g. {missing_stripes[0].name}")

3s: MSA files added for 60 sites
5s: MSA files added for 60 sites


## 5. Write the per-storey batch launchers

One launcher per storey at `BATCH_BASE/{n}s/mdof/msa_{IM}.py` (e.g.
`.../3s/mdof/msa_AvgSA03.py`) — the folder path encodes the storeys + system, so the
filename is just the analysis type. Each lists a job per built site: its
`run_msa_site.py` + `config_msa_{IM}.py`. When `FILTER_TO_COMPLETE_SITES` is `True`,
only sites in `ready_sites` are listed. `configure_batch_run_file` serialises these
into the runnable launcher (one console window per job).

In [ ]:
batch_name = f"msa_{GM_SET.replace('_', '')}.py"   # -> msa_AvgSA03.py

for n in STOREYS:
    jobs = [
        {
            "script": folder / "run_msa_site.py",
            "config": [folder / f"config_msa_{GM_SET}.py"],
            "name": [f"site_{site_idx}_{n}s_mdof_msa_{GM_SET}"],
        }
        for site_idx, folder in sorted(built[n].items())
        if not FILTER_TO_COMPLETE_SITES or site_idx in ready_sites
    ]
    batch_dir = BATCH_BASE / f"{n}s" / "mdof"
    batch_dir.mkdir(parents=True, exist_ok=True)
    batch_dst = batch_dir / batch_name
    configure_batch_run_file(cfg["templates"]["batch_run"], batch_dst, jobs)
    filt = "completed sites only" if FILTER_TO_COMPLETE_SITES else "all built sites"
    print(f"wrote {batch_dst.relative_to(BATCH_BASE)}: {len(jobs)} jobs ({filt})")